# Benchmark Analysis

This notebook is focused on the Whispbook speculative-decoding benchmark after source text has already been prepared as story fixtures.

Use it to load fixture data, inspect checkpoint choices, run or load benchmark results, compare modes, and inspect sweep metrics.

## Runtime Setup

Run this notebook from the repository root or from the `notebooks/` directory. Live MLX inference stays disabled unless `RUN_LIVE_MLX_GENDER=1` is set before launching the kernel.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        package_dir = candidate / "src" / "whispbook_gender_specdec"
        if (candidate / "pyproject.toml").exists() and package_dir.exists():
            return candidate
    raise RuntimeError("Could not find repository root from the current notebook path.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from whispbook_gender_specdec.benchmark import (  # noqa: E402
    ModeConfig,
    default_mode_configs,
    load_story_benchmark_dataframe,
    normalize_sample_dataframe,
    run_benchmark_dataframe,
    source_hint_from_sample_row,
    write_benchmark_csv,
)
from whispbook_gender_specdec.checkpoints import discover_model_specs  # noqa: E402
from whispbook_gender_specdec.gender import GenderInferenceResult  # noqa: E402
from whispbook_gender_specdec.mlx_runner import infer_with_mlx  # noqa: E402
from whispbook_gender_specdec.results import (  # noqa: E402
    summarize_results_by_speaker_dataframe,
    summarize_results_dataframe,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)
print(f"Project root: {PROJECT_ROOT}")

## Controls

Choose the fixture, sample count, model families, and output paths here. The notebook can analyze saved results without loading models.

In [ ]:
STORY = "story_2"
STORY_DIR = PROJECT_ROOT / "data" / STORY
SAMPLE_LIMIT = 8

SMALL_FAMILY = "qwen_small"
LARGE_FAMILY = "qwen_large"
LOCAL_ONLY = False

RUN_LIVE = os.environ.get("RUN_LIVE_MLX_GENDER") == "1"
MAX_TOKENS = 64
GAMMA = 7

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / STORY / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT_DIR / "benchmark_analysis_results.csv"
MODEL_COMPARISON_PREDICTIONS_CSV = OUTPUT_DIR / "model_comparison_predictions.csv"
SPECULATIVE_SWEEP_CSV = OUTPUT_DIR / "speculative_sweep.csv"

print(f"RUN_LIVE={RUN_LIVE}; set RUN_LIVE_MLX_GENDER=1 before launching Jupyter to run fresh MLX inference.")

## Load Story Fixture

The benchmark input is a labeled story fixture: plain text, speaker attribution rows, and optional metadata. Rows move through the notebook as DataFrames.

In [ ]:
def one_path(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching {pattern!r} in {directory}")
    return matches[0]


def fixture_paths(story_dir: Path) -> tuple[Path, Path, Path | None]:
    story_text_path = one_path(story_dir, "*_20000_word_story.txt")
    ground_truth_path = one_path(story_dir, "*_speaker_attribution_ground_truth.jsonl")
    metadata_matches = sorted(story_dir.glob("*_benchmark_metadata.json"))
    metadata_path = metadata_matches[0] if metadata_matches else None
    return story_text_path, ground_truth_path, metadata_path


def display_table(frame: pd.DataFrame, columns: list[str] | None = None, limit: int = 20) -> None:
    if frame.empty:
        display(Markdown("_No rows._"))
        return
    if columns is None:
        view = frame
    else:
        view = frame.loc[:, [column for column in columns if column in frame.columns]]
    display(view.head(limit))


STORY_TEXT_PATH, STORY_GROUND_TRUTH_PATH, STORY_METADATA_PATH = fixture_paths(STORY_DIR)
samples_df = load_story_benchmark_dataframe(
    ground_truth_path=STORY_GROUND_TRUTH_PATH,
    story_text_path=STORY_TEXT_PATH,
    metadata_path=STORY_METADATA_PATH,
)
samples_df = normalize_sample_dataframe(samples_df.head(SAMPLE_LIMIT).reset_index(drop=True))

source_name = STORY_DIR.relative_to(PROJECT_ROOT)
display(Markdown(f"Loaded `{source_name}` with {len(samples_df)} benchmark samples."))
display_table(
    samples_df,
    ["sample_id", "speaker", "expected_gender", "story_title", "text"],
    limit=SAMPLE_LIMIT,
)

## Fixture Shape

Use these quick summaries to check balance, speaker coverage, and text length before running model comparisons.

In [ ]:
fixture_profile = samples_df.assign(text_chars=samples_df["text"].str.len())

speaker_profile = (
    fixture_profile.groupby(["speaker", "expected_gender"], sort=True)
    .agg(samples=("sample_id", "size"), avg_text_chars=("text_chars", "mean"))
    .reset_index()
)
gender_profile = (
    fixture_profile.groupby("expected_gender", sort=True)
    .agg(samples=("sample_id", "size"), avg_text_chars=("text_chars", "mean"))
    .reset_index()
)

display(Markdown("**By speaker**"))
display_table(speaker_profile)
display(Markdown("**By expected gender**"))
display_table(gender_profile)

## Checkpoint Selection

The default mode set compares small-only, large-only, and speculative decoding with the configured model families.

In [ ]:

manifest = discover_model_specs(local_only=LOCAL_ONLY)
modes = default_mode_configs(
    manifest,
    small_family=SMALL_FAMILY,
    large_family=LARGE_FAMILY,
)

checkpoint_df = pd.DataFrame(
    [
        {
            "family": family,
            "label": spec.label,
            "model_ref": spec.model_ref or "",
            "local_path": str(spec.path) if spec.path else "",
        }
        for family, spec in sorted(manifest.items())
    ]
)
mode_df = pd.DataFrame(
    [
        {
            "mode": mode.name,
            "target": mode.target.family,
            "draft": mode.draft.family if mode.draft is not None else "",
        }
        for mode in modes
    ]
)

display(Markdown("**Available model families**"))
display_table(checkpoint_df)
display(Markdown("**Benchmark modes**"))
display_table(mode_df)

## Run Or Load Results

With live inference disabled, the notebook first reads `benchmark_analysis_results.csv`, then falls back to the existing model-comparison rows when present.

In [ ]:
def load_csv_frame(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def infer_one(mode: ModeConfig, sample: pd.Series, *, max_tokens: int, gamma: int) -> GenderInferenceResult:
    return infer_with_mlx(
        target=mode.target,
        draft=mode.draft,
        text=str(sample["text"]),
        source_hint=source_hint_from_sample_row(sample),
        max_tokens=max_tokens,
        num_draft_tokens=gamma,
    )


loaded_from: Path | None = None
if RUN_LIVE:
    results_df = run_benchmark_dataframe(
        samples=samples_df,
        modes=modes,
        infer=lambda mode, sample: infer_one(mode, sample, max_tokens=MAX_TOKENS, gamma=GAMMA),
        parameters={"max_tokens": MAX_TOKENS, "num_draft_tokens": GAMMA},
    )
    write_benchmark_csv(results_df, RESULTS_CSV)
    loaded_from = RESULTS_CSV
else:
    results_df = load_csv_frame(RESULTS_CSV)
    if not results_df.empty:
        loaded_from = RESULTS_CSV
    else:
        results_df = load_csv_frame(MODEL_COMPARISON_PREDICTIONS_CSV)
        if not results_df.empty:
            loaded_from = MODEL_COMPARISON_PREDICTIONS_CSV

if loaded_from is None:
    display(Markdown("No saved result rows found. Enable live inference or run a benchmark command first."))
else:
    display(Markdown(f"Loaded {len(results_df)} result rows from `{loaded_from.relative_to(PROJECT_ROOT)}`."))

display_table(results_df, limit=10)

## Mode Summary

This is the main benchmark rollup: accuracy, parse quality, mean latency, and draft acceptance.

In [ ]:
summary_df = summarize_results_dataframe(results_df)
speaker_summary_df = summarize_results_by_speaker_dataframe(results_df)

display(Markdown("**Overall mode summary**"))
display_table(summary_df)
display(Markdown("**By speaker**"))
display_table(speaker_summary_df, limit=30)

## Visual Summary

The charts below make mode-level latency and quality tradeoffs easier to scan.

In [ ]:
def plot_summary(frame: pd.DataFrame) -> None:
    if frame.empty:
        display(Markdown("No summary rows to plot."))
        return
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        display(Markdown("Install `matplotlib` to render charts."))
        return

    metrics = [
        ("accuracy", "Accuracy"),
        ("avg_wall_ms", "Avg wall time ms"),
        ("avg_draft_acceptance_ratio", "Draft acceptance"),
    ]
    fig, axes = plt.subplots(1, len(metrics), figsize=(14, 4), constrained_layout=True)
    for axis, (column, title) in zip(axes, metrics):
        axis.bar(frame["mode"], pd.to_numeric(frame[column], errors="coerce"))
        axis.set_title(title)
        axis.tick_params(axis="x", rotation=25)
        axis.grid(axis="y", alpha=0.25)
    plt.show()


plot_summary(summary_df)

## Miss Analysis

Inspect incorrect rows first. Most benchmark surprises come from label ambiguity, truncated JSON, or mode-specific response parsing.

In [ ]:
def miss_frame(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty or "correct" not in results.columns:
        return pd.DataFrame()
    frame = results.copy()
    if frame["correct"].dtype != bool:
        frame["correct"] = frame["correct"].map(lambda value: str(value).strip().lower() in {"1", "true", "t", "yes", "y"})
    return frame.loc[~frame["correct"]].copy()


misses_df = miss_frame(results_df)
miss_columns = [
    "mode",
    "sample_id",
    "speaker",
    "expected_gender",
    "predicted_gender",
    "confidence",
    "evidence",
    "parse_ok",
    "wall_ms",
]
display_table(misses_df, miss_columns, limit=30)

## Model Comparison

When the loaded result rows include multiple target or draft families, this table separates model effects from mode effects.

In [ ]:
def model_comparison(results: pd.DataFrame) -> pd.DataFrame:
    required = {"mode", "target_family", "draft_family", "correct", "wall_ms"}
    if results.empty or not required.issubset(results.columns):
        return pd.DataFrame()
    frame = results.copy()
    if frame["correct"].dtype != bool:
        frame["correct"] = frame["correct"].map(lambda value: str(value).strip().lower() in {"1", "true", "t", "yes", "y"})
    if "draft_acceptance_ratio" not in frame.columns:
        frame["draft_acceptance_ratio"] = 0.0
    return (
        frame.groupby(["mode", "target_family", "draft_family"], dropna=False, sort=True)
        .agg(
            samples=("sample_id", "size"),
            accuracy=("correct", "mean"),
            avg_wall_ms=("wall_ms", "mean"),
            avg_draft_acceptance_ratio=("draft_acceptance_ratio", "mean"),
        )
        .reset_index()
    )


model_comparison_df = model_comparison(results_df)
display_table(model_comparison_df, limit=40)

## Sweep Analysis

If a sweep file exists, use it to inspect how `gamma` and `max_tokens` change accuracy, latency, and draft acceptance.

In [ ]:
sweep_df = load_csv_frame(SPECULATIVE_SWEEP_CSV)

def plot_sweep(frame: pd.DataFrame) -> None:
    if frame.empty:
        display(Markdown("No sweep rows found."))
        return
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        display(Markdown("Install `matplotlib` to render charts."))
        display_table(frame)
        return

    frame = frame.copy()
    for column in ["gamma", "max_tokens", "accuracy", "avg_wall_ms", "avg_draft_acceptance_ratio"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    metrics = [
        ("accuracy", "Accuracy"),
        ("avg_wall_ms", "Avg wall time ms"),
        ("avg_draft_acceptance_ratio", "Draft acceptance"),
    ]
    fig, axes = plt.subplots(1, len(metrics), figsize=(15, 4), constrained_layout=True)
    for axis, (metric, title) in zip(axes, metrics):
        for max_tokens, group in frame.groupby("max_tokens"):
            ordered = group.sort_values("gamma")
            axis.plot(ordered["gamma"], ordered[metric], marker="o", label=f"max_tokens={max_tokens}")
        axis.set_title(title)
        axis.set_xlabel("gamma")
        axis.grid(True, alpha=0.25)
    axes[-1].legend(loc="best")
    plt.show()


display_table(sweep_df, limit=20)
plot_sweep(sweep_df)

## Interpretation Checklist

- Compare `large_only` against `speculative` first; speculative should preserve target behavior when tokenizers and runner wiring are compatible.
- Treat tiny sample sets as wiring checks, not production latency evidence.
- Low parse quality usually means `MAX_TOKENS` is too small or the prompt is not constraining the response enough.
- Low draft acceptance means the draft model is not close enough for this prompt style; lower `GAMMA` or choose a closer draft checkpoint.
- Report row-level misses with raw responses before changing labels or prompts.